In [0]:
df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("abfss://data@monitoringpipeline.dfs.core.windows.net/Raw_data/server_logs_raw.csv")

display(df)

Server_ID,Hostname,IP_Address,OS_Type,Server_Location,CPU_Utilization (%),Memory_Usage (%),Disk_IO (%),Network_Traffic_In (MB/s),Network_Traffic_Out (MB/s),Uptime (Hours),Downtime (Hours),Admin_Name,Admin_Email,Admin_Phone,Log_Timestamp
536c9e75-40a1-4a74-8ca6-c6abaec13999,srv-web02,100.33.3.46,Windows Server 2022,"Berlin, Germany",28.1,88.6,28.36,31.13,52.54,18.49,5.51,John Duarte,laurasolomon@example.net,+77 1834560143,11-03-25 16:45
26bae71d-c025-4082-9484-ef7aa9e04836,srv-web02,249.168.171.15,Ubuntu 20.04,"London, UK",49.72,84.35,73.89,123.34,97.1,19.71,4.29,Abigail Montgomery,sherripatterson@example.com,+10 3791349757,25-01-25 10:59
454de38a-5961-490f-9d46-d20663a36baf,srv-backup02,248.247.52.48,Red Hat Enterprise Linux,"New York, USA",44.72,57.64,14.94,69.6,112.81,18.64,5.36,David Jones,veronicanavarro@example.net,+38 9503876034,29-01-25 08:31
8b8a3f40-1b67-4fe6-a65e-93fd07f317f7,srv-web02,87.221.187.138,Red Hat Enterprise Linux,"New York, USA",37.69,97.97,71.46,62.97,148.51,18.14,5.86,Dominic Richards,frankdawson@example.org,+93 2330701744,01-03-25 16:49
375cb3db-89d3-4705-af4a-8f829cce1928,srv-app02,168.8.91.217,Windows Server 2022,"Tokyo, Japan",63.97,67.31,96.46,45.33,131.5,22.78,1.22,Paul Edwards,hgibson@example.com,+29 9075330226,21-02-25 11:02
9a90becd-5be4-4d03-b1fa-1ffa70cb11f5,srv-app02,142.229.13.87,Ubuntu 20.04,"Tokyo, Japan",20.29,24.65,66.56,60.59,133.97,18.68,5.32,Thomas Pearson,monicamartinez@example.org,+41 5610383316,13-02-25 19:44
c2ba6509-caa4-4720-8633-cb7565e9947c,srv-app03,143.148.212.7,Ubuntu 20.04,"Sydney, Australia",36.45,59.28,18.46,47.7,86.57,22.38,1.62,Richard Goodman,sarah65@example.com,+77 4103444382,09-03-25 14:58
8e52d19d-1e84-466e-8e25-d24ae27434ab,srv-web01,147.18.138.45,Ubuntu 20.04,"Tokyo, Japan",12.62,86.98,18.54,102.76,11.03,20.3,3.7,Jessica Day,sheri42@example.org,+75 8001044687,06-03-25 10:06
95de17b4-60ec-4ea0-a79c-7faecdb85275,srv-app03,157.114.125.79,CentOS 8,"Sydney, Australia",75.49,73.48,93.03,29.02,22.0,20.49,3.51,Stephanie Petersen,kevin33@example.org,+71 9513511325,24-02-25 14:29
4aa1fa83-c575-4d81-a1c7-f1ed5395a0f6,srv-analytics01,186.179.201.0,CentOS 8,"Tokyo, Japan",88.3,24.14,25.99,67.86,120.95,20.19,3.81,Aaron Marquez MD,darrenklein@example.net,+98 6695197615,20-02-25 18:41


In [0]:
df.printSchema()
df.columns

root
 |-- Server_ID: string (nullable = true)
 |-- Hostname: string (nullable = true)
 |-- IP_Address: string (nullable = true)
 |-- OS_Type: string (nullable = true)
 |-- Server_Location: string (nullable = true)
 |-- CPU_Utilization (%): double (nullable = true)
 |-- Memory_Usage (%): double (nullable = true)
 |-- Disk_IO (%): double (nullable = true)
 |-- Network_Traffic_In (MB/s): double (nullable = true)
 |-- Network_Traffic_Out (MB/s): double (nullable = true)
 |-- Uptime (Hours): double (nullable = true)
 |-- Downtime (Hours): double (nullable = true)
 |-- Admin_Name: string (nullable = true)
 |-- Admin_Email: string (nullable = true)
 |-- Admin_Phone: string (nullable = true)
 |-- Log_Timestamp: string (nullable = true)



['Server_ID',
 'Hostname',
 'IP_Address',
 'OS_Type',
 'Server_Location',
 'CPU_Utilization (%)',
 'Memory_Usage (%)',
 'Disk_IO (%)',
 'Network_Traffic_In (MB/s)',
 'Network_Traffic_Out (MB/s)',
 'Uptime (Hours)',
 'Downtime (Hours)',
 'Admin_Name',
 'Admin_Email',
 'Admin_Phone',
 'Log_Timestamp']

In [0]:
df_clean = df.toDF(*[
    c.replace(" ", "_")
     .replace("(", "")
     .replace(")", "")
     .replace("%", "")
     .replace("/", "_")
     .strip("_")
    for c in df.columns
])

df_clean.columns

['Server_ID',
 'Hostname',
 'IP_Address',
 'OS_Type',
 'Server_Location',
 'CPU_Utilization',
 'Memory_Usage',
 'Disk_IO',
 'Network_Traffic_In_MB_s',
 'Network_Traffic_Out_MB_s',
 'Uptime_Hours',
 'Downtime_Hours',
 'Admin_Name',
 'Admin_Email',
 'Admin_Phone',
 'Log_Timestamp']

In [0]:
df_clean = df_clean.dropna()
df_clean = df_clean.dropDuplicates()

In [0]:
from pyspark.sql.functions import col

numeric_cols = [
"CPU_Utilization",
"Memory_Usage",
"Disk_IO",
"Network_Traffic_In_MB_s",
"Network_Traffic_Out_MB_s",
"Uptime_Hours",
"Downtime_Hours"
]

for column in numeric_cols:
    df_clean = df_clean.withColumn(column, col(column).cast("double"))

In [0]:
from pyspark.sql.functions import to_timestamp, col

df_clean = df_clean.withColumn(
    "Log_Timestamp",
    to_timestamp(col("Log_Timestamp"), "yy-MM-dd HH:mm")
)

In [0]:
df_clean = df_clean.filter(
    (col("CPU_Utilization") >= 0) &
    (col("CPU_Utilization") <= 100)
)

In [0]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "Server_Health_Status",
    when((col("CPU_Utilization") > 85) | (col("Memory_Usage") > 85), "Critical")
    .when(col("CPU_Utilization") > 60, "Warning")
    .otherwise("Healthy")
)

In [0]:
df_clean = df_clean.withColumn(
    "Resource_Utilization_Score",
    (col("CPU_Utilization") + col("Memory_Usage") + col("Disk_IO")) / 3
)

In [0]:
df_clean = df_clean.withColumn(
    "Availability_Percentage",
    (col("Uptime_Hours") /
    (col("Uptime_Hours") + col("Downtime_Hours"))) * 100
)

In [0]:
from pyspark.sql.functions import sha2

df_clean = df_clean.withColumn(
    "IP_Address_Hash",
    sha2(col("IP_Address"),256)
)

In [0]:
df_clean = df_clean.drop("IP_Address")

In [0]:
from pyspark.sql.functions import avg

server_metrics = df_clean.groupBy("Server_ID").agg(
    avg("CPU_Utilization").alias("Avg_CPU_Utilization"),
    avg("Memory_Usage").alias("Avg_Memory_Usage"),
    avg("Disk_IO").alias("Avg_Disk_IO")
)

display(server_metrics)

Server_ID,Avg_CPU_Utilization,Avg_Memory_Usage,Avg_Disk_IO
ad737c08-386f-4394-9ad2-76c244861bb8,46.58,55.13,50.7
3145367a-2b21-41fc-8eef-1d92088de8a4,88.99,42.49,91.11
a23db8da-5330-4089-9a05-4ac1390954b7,34.04,91.66,33.55
ed4746e9-fd4e-49b8-b3ae-067cab9c1c43,59.34,54.8,89.17
961a2a06-4fd0-485d-bd56-d4e52ab5039c,17.7,97.15,63.26
f0dc3a9a-113e-4f4a-a272-3338a300ab00,48.93,68.43,82.21
a8fcbe97-16a7-4408-a2ad-dfbd2c531107,45.87,72.67,61.55
6a035308-8cef-404d-9a17-eb294ecf8b43,19.83,92.92,86.36
a6212f18-0993-4f4f-ab4d-6fc3452bde64,55.46,83.42,61.89
26bae71d-c025-4082-9484-ef7aa9e04836,49.72,84.35,73.89


In [0]:
from pyspark.sql.functions import sum

downtime_summary = df_clean.groupBy("Server_ID").agg(
    sum("Downtime_Hours").alias("Total_Downtime_Hours")
)

display(downtime_summary)

Server_ID,Total_Downtime_Hours
ad737c08-386f-4394-9ad2-76c244861bb8,4.45
3145367a-2b21-41fc-8eef-1d92088de8a4,3.43
a23db8da-5330-4089-9a05-4ac1390954b7,4.9
ed4746e9-fd4e-49b8-b3ae-067cab9c1c43,2.2
961a2a06-4fd0-485d-bd56-d4e52ab5039c,3.28
f0dc3a9a-113e-4f4a-a272-3338a300ab00,2.06
a8fcbe97-16a7-4408-a2ad-dfbd2c531107,5.99
6a035308-8cef-404d-9a17-eb294ecf8b43,5.32
a6212f18-0993-4f4f-ab4d-6fc3452bde64,2.78
26bae71d-c025-4082-9484-ef7aa9e04836,4.29


In [0]:
health_distribution = df_clean.groupBy("Server_Health_Status").count()

display(health_distribution)

Server_Health_Status,count
Healthy,82
Critical,73
Warning,45


In [0]:
network_metrics = df_clean.groupBy("Server_ID").agg(
    avg("Network_Traffic_In_MB_s").alias("Avg_Network_In"),
    avg("Network_Traffic_Out_MB_s").alias("Avg_Network_Out")
)

display(network_metrics)

Server_ID,Avg_Network_In,Avg_Network_Out
ad737c08-386f-4394-9ad2-76c244861bb8,18.46,10.27
3145367a-2b21-41fc-8eef-1d92088de8a4,132.52,35.24
a23db8da-5330-4089-9a05-4ac1390954b7,43.35,91.64
ed4746e9-fd4e-49b8-b3ae-067cab9c1c43,57.42,59.71
961a2a06-4fd0-485d-bd56-d4e52ab5039c,118.09,130.34
f0dc3a9a-113e-4f4a-a272-3338a300ab00,125.12,73.54
a8fcbe97-16a7-4408-a2ad-dfbd2c531107,142.08,147.92
6a035308-8cef-404d-9a17-eb294ecf8b43,116.42,100.22
a6212f18-0993-4f4f-ab4d-6fc3452bde64,90.53,25.47
26bae71d-c025-4082-9484-ef7aa9e04836,123.34,97.1


In [0]:
from pyspark.sql.functions import year, month, dayofmonth, hour

df_clean = df_clean.withColumn("Year", year("Log_Timestamp"))
df_clean = df_clean.withColumn("Month", month("Log_Timestamp"))
df_clean = df_clean.withColumn("Day", dayofmonth("Log_Timestamp"))
df_clean = df_clean.withColumn("Hour", hour("Log_Timestamp"))

In [0]:
# Write cleaned dataset
df_clean.write \
.mode("overwrite") \
.option("header","true") \
.csv("abfss://data@monitoringpipeline.dfs.core.windows.net/transformed_data/server_logs_clean")

# Server metrics
server_metrics.write \
.mode("overwrite") \
.option("header","true") \
.csv("abfss://data@monitoringpipeline.dfs.core.windows.net/transformed_data/server_metrics")

# Downtime summary
downtime_summary.write \
.mode("overwrite") \
.option("header","true") \
.csv("abfss://data@monitoringpipeline.dfs.core.windows.net/transformed_data/downtime_summary")

# Network metrics
network_metrics.write \
.mode("overwrite") \
.option("header","true") \
.csv("abfss://data@monitoringpipeline.dfs.core.windows.net/transformed_data/network_metrics")

# Health distribution
health_distribution.write \
.mode("overwrite") \
.option("header","true") \
.csv("abfss://data@monitoringpipeline.dfs.core.windows.net/transformed_data/health_distribution")